# 📊 Asian Options & Volatility Surfaces (Pandas)
Asian options are path-dependent. Their payoff depends on the arithmetic average of the asset's trajectory over time. This requires simulating the Stochastic Differential Equation (SDE) step-by-step.

> **API Key Required:** Generating a volatility surface requires launching multiple concurrent matrices that exceed the public demo limits. 
> Grab your private API Key and **50 Free Credits** at **[prometheusquantengine.com](https://prometheusquantengine.com)** to execute this batch.

### ⚙️ The Mathematical Load
* We will use `m_steps = 252` (daily observations over 1 year).
* We will dispatch a batch of 10 matrices across different volatilities and strikes.
* **Cost per Node:** N=100,000 × M=252 = 25.2 Million Steps = **~0.10 Credits**.

In [ ]:
import requests
import uuid
import pandas as pd
import time

API_KEY = "pmt_live_..." # 👈 PASTE YOUR PRIVATE API KEY HERE
BASE_URL = "https://api.prometheusquantengine.com/api/v1/simulations"

# Define the surface parameters
volatilities = [0.15, 0.20, 0.25, 0.30, 0.35]
strikes = [90.0, 100.0]
results = []

print("Igniting C++ OpenMP cores. Dispatching 10 synchronous matrices...\n")
start_batch = time.time()

for vol in volatilities:
    for k in strikes:
        payload = {
            "simulation_type": "Asian",
            "s_0": 100.0,
            "strike": k,
            "volatility": vol,
            "time_to_maturity": 1.0,
            "risk_free_rate": 0.05,
            "option_type": "Call",
            "n_simulations": 100000,
            "m_steps": 252
        }
        
        headers = {"X-API-Key": API_KEY, "Idempotency-Key": str(uuid.uuid4())}
        resp = requests.post(BASE_URL, json=payload, headers=headers)
        
        if resp.status_code == 403:
            print("❌ Error: Please insert a valid API Key from the dashboard.")
            break
            
        data = resp.json()
        results.append({
            "Strike": k,
            "Volatility": f"{vol*100} %",
            "Fair Value": round(data.get('fair_value', 0), 4),
            "Gamma (Γ)": round(data.get('gamma', 0), 6),
            "Compute Latency (s)": round(time.time() - start_batch, 4)
        })

if results:
    df = pd.DataFrame(results)
    print(f"Batch computation completed in {time.time() - start_batch:.2f} seconds.")
    display(df) # Renders a formatted DataFrame in Colab

### 🏛️ Institutional Integration
By utilizing the API, you bypassed the Global Interpreter Lock (GIL) of Python, evaluating over **250 Million stochastic steps** in seconds without locking your local hardware. 

For massive models exceeding 50M steps per request, the API utilizes Asynchronous Polling, demonstrated in **Notebook 03**.